In [ ]:
# ============================================
# RUN THIS FIRST — Colab Setup
# ============================================
!pip install datasets feedparser sentence-transformers -q

# Clone the repo to access project structure
!git clone https://github.com/melissanoarianna1-commits/finpulse-ai.git 2>/dev/null || echo 'Repo already cloned'
import os
os.chdir('/content/finpulse-ai')
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/validated', exist_ok=True)
os.makedirs('models/sentiment', exist_ok=True)
os.makedirs('models/recommender', exist_ok=True)
os.makedirs('docs', exist_ok=True)
print('✅ Setup complete. Working directory:', os.getcwd())

# Model 1: Financial Sentiment Classifier

## Overview
Fine-tunes **FinBERT** (BERT pre-trained on financial text) on the **Financial PhraseBank** dataset
to classify financial sentences as **positive**, **negative**, or **neutral**.

## Role in FinPulse AI
Powers the **sentiment dots** (green/red/gray) and **impact badges** on news cards in the dashboard.

## Connection to Financial Risk
- **Market risk**: Negative sentiment predicts short-term price drops
- **Credit risk**: Negative press about a borrower is an early warning signal
- **Operational risk**: Sentiment monitoring detects reputational risk events

## Approach
1. Load pre-trained FinBERT from HuggingFace
2. Tokenize Financial PhraseBank
3. Fine-tune for 3 epochs
4. Evaluate on validation set (accuracy, F1, confusion matrix)

## 1. Load & Explore Data

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load data
dataset = load_dataset('financial_phrasebank', 'sentences_allagree', trust_remote_code=True)
df = pd.DataFrame(dataset['train'])
label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
df['label_text'] = df['label'].map(label_names)

print(f'Dataset: {len(df)} sentences')
print(df['label_text'].value_counts())

In [ ]:
# Visualize label distribution
fig, ax = plt.subplots(figsize=(8, 4))
colors = {'negative': '#ef4444', 'neutral': '#64748b', 'positive': '#10b981'}
counts = df['label_text'].value_counts()
ax.bar(counts.index, counts.values, color=[colors[l] for l in counts.index])
ax.set_title('Financial PhraseBank — Label Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of sentences')
for i, (label, count) in enumerate(counts.items()):
    ax.text(i, count + 10, str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('docs/sentiment_label_distribution.png', dpi=150)
plt.show()

## 2. Train/Validation Split

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f'Training: {len(train_df)} | Validation: {len(val_df)}')

train_dataset = Dataset.from_pandas(train_df[['sentence', 'label']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['sentence', 'label']].reset_index(drop=True))

## 3. Tokenize with FinBERT

FinBERT uses the same tokenizer as BERT but was pre-trained on financial text
(10-K filings, analyst reports, financial news). This domain-specific pre-training means
it already understands financial language before we fine-tune it.

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['sentence'], padding='max_length', truncation=True, max_length=128)

train_tok = train_dataset.map(tokenize_fn, batched=True)
val_tok = val_dataset.map(tokenize_fn, batched=True)
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print('✅ Tokenization complete')

## 4. Fine-Tune FinBERT

Training config:
- **Epochs**: 3 (standard for BERT — more risks overfitting on small datasets)
- **Batch size**: 16
- **Learning rate**: 2e-5 (standard BERT fine-tuning rate)
- **Evaluation**: Every epoch, keeping best model by F1

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir='models/sentiment',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=25,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()
print('✅ Training complete!')

## 5. Evaluate on Validation Set

In [ ]:
predictions = trainer.predict(val_tok)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

target_names = ['negative', 'neutral', 'positive']
print('='*60)
print('CLASSIFICATION REPORT — Validation Set')
print('='*60)
print(classification_report(labels, preds, target_names=target_names))
print(f'Overall Accuracy: {accuracy_score(labels, preds):.4f}')
print(f'Macro F1 Score:   {f1_score(labels, preds, average="macro"):.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Sentiment Model — Confusion Matrix (Validation)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/sentiment_confusion_matrix.png', dpi=150)
plt.show()

## 6. Test on Custom Financial Sentences

In [ ]:
test_sentences = [
    'The bank reported record profits in Q4, exceeding analyst expectations.',
    'Credit losses surged 40% as loan defaults increased across all segments.',
    'The ECB maintained interest rates at current levels as expected.',
    'Deutsche Bank announced 3,500 job cuts amid restructuring efforts.',
    'Strong capital ratios and improved liquidity position the bank well for 2026.',
]

print('Custom Predictions:')
print('='*80)
for sentence in test_sentences:
    inputs = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=128, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_idx = probs.argmax().item()
    print(f'\n  "{sentence}"')
    print(f'  → {target_names[pred_idx].upper()} ({probs[pred_idx]:.1%})')
    print(f'    [neg: {probs[0]:.1%} | neu: {probs[1]:.1%} | pos: {probs[2]:.1%}]')

## 7. Save Model

In [ ]:
model.save_pretrained('models/sentiment/finbert-finpulse')
tokenizer.save_pretrained('models/sentiment/finbert-finpulse')
print('✅ Model saved to models/sentiment/finbert-finpulse/')
print('\n--- Model 1 Complete ---')
print('Next: Run 02_recommender_model.ipynb')